# Module 3: Data Cleaning

## Overview

This notebook implements the **Data Cleaning** module of the TrustGuard project.

The dataset is cleaned by handling missing values, removing duplicate records, standardizing text and date formats, performing data type conversions, and preparing a clean dataset. The processed data is stored as a Delta Table for analytical use.

In [0]:
# Import required PySpark functions and load Raw Layer

from pyspark.sql.functions import *
from pyspark.sql.window import Window

raw_df = spark.table("trustguard.raw_transactions")

display(raw_df.limit(10))

transaction_id,customer_id,category,item,price_per_unit,quantity,total_spent,payment_method,location,transaction_date,discount_applied
TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,true
TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,true
TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,false
TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,null
TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,false
TXN_7482416,CUST_09,Patisserie,null,null,10.0,200.0,Credit Card,Online,2023-11-30,null
TXN_3652209,CUST_07,Food,Item_1_FOOD,5.0,8.0,40.0,Credit Card,In-store,2023-06-10,true
TXN_1372952,CUST_21,Furniture,null,33.5,null,null,Digital Wallet,In-store,2024-04-02,true
TXN_9728486,CUST_23,Furniture,Item_16_FUR,27.5,1.0,27.5,Credit Card,In-store,2023-04-26,false
TXN_2722661,CUST_25,Butchers,Item_22_BUT,36.5,3.0,109.5,Cash,Online,2024-03-14,false


In [0]:
# Reject records where transaction_id or customer_id is missing

rejected_df = (
    raw_df
    .filter(
        col("transaction_id").isNull() |
        (trim(col("transaction_id")) == "") |
        col("customer_id").isNull() |
        (trim(col("customer_id")) == "")
    )
    .withColumn(
        "rejection_reason",
        lit("Missing transaction_id or customer_id")
    )
)

display(rejected_df)

transaction_id,customer_id,category,item,price_per_unit,quantity,total_spent,payment_method,location,transaction_date,discount_applied,rejection_reason


In [0]:
# Keep records with valid IDs and fill non-critical NULL values

clean_df = (
    raw_df
    .filter(
        col("transaction_id").isNotNull() &
        (trim(col("transaction_id")) != "") &
        col("customer_id").isNotNull() &
        (trim(col("customer_id")) != "")
    )
    .fillna({
        "category": "Unknown",
        "item": "Unknown",
        "payment_method": "Unknown",
        "location": "Unknown",
        "discount_applied": "No"
    })
)

In [0]:
# Convert columns into correct data types

clean_df = (
    clean_df
    .withColumn("price_per_unit", col("price_per_unit").cast("double"))
    .withColumn("quantity", col("quantity").cast("integer"))
    .withColumn("total_spent", col("total_spent").cast("double"))
)

In [0]:
# Standardize date and text columns

clean_df = (
    clean_df
    .withColumn(
        "transaction_date",
        coalesce(
            to_date(col("transaction_date"), "yyyy-MM-dd"),
            to_date(col("transaction_date"), "dd-MM-yyyy"),
            to_date(col("transaction_date"), "MM/dd/yyyy"),
            to_date(col("transaction_date"), "dd/MM/yyyy")
        )
    )
    .withColumn("category", initcap(trim(col("category"))))
    .withColumn("item", initcap(trim(col("item"))))
    .withColumn("payment_method", initcap(trim(col("payment_method"))))
    .withColumn("location", initcap(trim(col("location"))))
    .withColumn("discount_applied", initcap(trim(col("discount_applied"))))
)

In [0]:
from pyspark.sql.functions import avg

avg_values = clean_df.select(
    avg("price_per_unit").alias("avg_price"),
    avg("quantity").alias("avg_quantity"),
    avg("total_spent").alias("avg_total")
).first()

clean_df = clean_df.fillna({
    "price_per_unit": float(avg_values["avg_price"]),
    "quantity": int(avg_values["avg_quantity"]),
    "total_spent": float(avg_values["avg_total"])
})

In [0]:
# Remove duplicate transactions

before_count = clean_df.count()

clean_df = clean_df.dropDuplicates(["transaction_id"])

after_count = clean_df.count()

print("Rows Before Deduplication:", before_count)
print("Rows After Deduplication:", after_count)
print("Duplicates Removed:", before_count - after_count)

Rows Before Deduplication: 12575
Rows After Deduplication: 12575
Duplicates Removed: 0


In [0]:
# Save Clean Data as CSV

clean_df.coalesce(1) \
.write \
.mode("overwrite") \
.option("header", "true") \
.csv("/Volumes/workspace/default/trustguard_data/clean_data")